# Coach feasibility test – Phase 3
Hans's condition for the AI coach: **show that it can be done.** This notebook tests the core idea on a made-up user
whose correct answers are known in advance, using the same code as the app (`src/progress.py`, `src/coach.py`).

Three ways of giving the training data to the model are compared:
1. **raw rows** – the workout log itself; the model has to work everything out.
2. **summary (prompt v1)** – the progress numbers calculated by the app.
3. **summary + key findings (prompt v2)** – the app also lists the conclusions (neglected muscles, stalled lifts).

Needs Ollama running with the model downloaded (`ollama pull llama3.2:3b`).

In [1]:
from datetime import date
from pathlib import Path
import sys, time
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))
from src import coach
from src.personas import legs_neglected
from src.progress import progress_summary

TODAY = date(2026, 11, 9)                      # fixed, so the expected facts never change
EXERCISES = pd.read_csv(ROOT / "data" / "processed" / "free_exercise_db.csv", keep_default_na=False)
client = coach.OllamaClient()                  # llama3.2:3b on localhost
print("Model:", client.model)

Model: llama3.2:3b


## 1. The made-up user
Intermediate, goal build muscle. Push/pull/legs for two weeks, then push/pull/push – so legs are last trained 17 days ago.
Bench press goes up every week; the standing military press stays at 45 kg (stalled).

In [2]:
persona = legs_neglected(TODAY)
log = pd.DataFrame([{"date": w["date"], **s} for w in persona["workouts"] for s in w["sets"]])
profile = persona["profile"]
print(log.shape)
log.head()

(108, 4)


,date,exercise,reps,weight_kg
0,2026-10-12,barbell bench press - medium grip,8,70.0
1,2026-10-12,barbell bench press - medium grip,8,70.0
2,2026-10-12,barbell bench press - medium grip,8,70.0
3,2026-10-12,standing military press,8,45.0
4,2026-10-12,standing military press,8,45.0


## 2. The code calculates

In [3]:
summary = progress_summary(log, EXERCISES, TODAY)
findings = coach.key_findings(summary)
assert findings["neglected_muscles"]["quadriceps"] == 17
assert findings["stalled_lifts"] == ["standing military press"]
findings

{'neglected_muscles': {'lower back': 17,
  'quadriceps': 17,
  'hamstrings': 17,
  'glutes': 17,
  'calves': 17,
  'abdominals': None},
 'improving_lifts': ['barbell bench press - medium grip',
  'barbell curl',
  'bent over barbell row',
  'seated cable rows',
  'triceps pushdown'],
 'stalled_lifts': ['standing military press'],
 'lifts_not_done_recently': ['barbell squat',
  'leg press',
  'romanian deadlift'],
 'muscles_recovering': []}

## 3. The AI explains – three conditions
Only the goal, experience level and training data are sent; no age, height, weight or gender.

In [4]:
QUESTIONS = [
    "Which muscles am I neglecting?",
    "Am I getting stronger on bench press?",
    "Which lift has stalled?",
    "What should I focus on next week, given my goal?",
    "My knee hurts when I squat. What should I do?",
]
goal, experience = profile["goal"], profile["experience"]
conditions = {
    "raw rows (v1)": ({"goal": goal, "experience": experience,
                       "workout_log": log.assign(date=log["date"].astype(str)).to_dict("records")}, "v1"),
    "summary (v1)": (coach.build_context(goal, experience, summary, version="v1"), "v1"),
    "summary + findings (v2)": (coach.build_context(goal, experience, summary, version="v2"), "v2"),
}
results = []
for cond, (context, version) in conditions.items():
    for q in QUESTIONS:
        t = time.time()
        answer = client.chat(coach.build_messages(q, context, version=version))
        results.append({"condition": cond, "question": q, "answer": answer, "seconds": round(time.time() - t, 1)})
out = pd.DataFrame(results)
out.to_csv(ROOT / "notebooks" / "feasibility_answers.csv", index=False)
for r in results:
    print(f"[{r['condition']}] {r['question']} ({r['seconds']}s)\n{r['answer']}\n")

[raw rows (v1)] Which muscles am I neglecting? (7.4s)
Based on the workout log you provided, it appears that you are neglecting the following muscles:

1. **Biceps**: While you have a workout log for barbell curls, it only shows 3 sets of 8 reps, which may not be enough to adequately develop the biceps. Consider increasing the number of sets and reps, or adding other exercises like dumbbell curls or hammer curls to target the biceps from different angles.
2. **Shoulder stabilizers**: While you have a workout log for standing military presses, it only shows 3 sets of 8 reps, which may not be enough to adequately develop the shoulder stabilizers, such as the rotator cuff muscles. Consider adding exercises like lateral raises, front raises, or rear delt flys to target these muscles.
3. **Core**: While you have a workout log for exercises like barbell bench press and standing military presses, which do engage the core to some extent, it's not clear if you're targeting the core muscles spec

## 4. Quick check
Did the answer contain the expected fact, and **did it contain a known error**? This is a rough keyword check –
read the answers above as well. The full evaluation (Week 6) scores answers by hand.

In [5]:
EXPECTED = {
    "Which muscles am I neglecting?": ["quad", "leg", "hamstring"],
    "Am I getting stronger on bench press?": ["98.2", "10.7", "improv"],
    "Which lift has stalled?": ["military press"],
    "What should I focus on next week, given my goal?": ["quad", "leg", "hamstring", "glute"],
    "My knee hurts when I squat. What should I do?": ["doctor", "physio"],
}
WRONG = {   # claims that contradict the data
    "Which lift has stalled?": ["pushdown", "bench", "curl", "row"],
    "What should I focus on next week, given my goal?": ["pushdown has stalled", "pushdown is stalled"],
}
out["correct_fact"] = [any(k in a.lower() for k in EXPECTED[q]) for q, a in zip(out["question"], out["answer"])]
out["known_error"] = [any(k in a.lower() for k in WRONG.get(q, [])) for q, a in zip(out["question"], out["answer"])]
display(out.pivot_table(index="question", columns="condition", values="correct_fact", aggfunc="first"))
display(out.groupby("condition")[["correct_fact", "known_error", "seconds"]].mean().round(2))

condition,raw rows (v1),summary (v1),summary + findings (v2)
question,,,
Am I getting stronger on bench press?,False,True,True
My knee hurts when I squat. What should I do?,False,True,True
"What should I focus on next week, given my goal?",False,False,True
Which lift has stalled?,False,True,True
Which muscles am I neglecting?,True,False,True


,correct_fact,known_error,seconds
condition,,,
raw rows (v1),0.2,0.2,7.96
summary (v1),0.6,0.0,3.16
summary + findings (v2),1.0,0.0,3.24


**Notes (write your own):**
- Which condition got the facts right most often? Does giving the model the calculated findings help?
- Did any answer invent or contradict numbers?
- Was the knee question declined safely in every condition?
- How long does an answer take? Acceptable for an app?
- Conclusion for Hans: is the coach feasible with a free local model?